# Cuaderno asociado al TFG

Este cuaderno forma parte del repositorio asociado al Trabajo de Fin de Grado **“Impacto de las Telecomunicaciones en la Agricultura 4.0”**.

Por motivos de confidencialidad, los archivos de datos reales no se incluyen en el repositorio. El código se mantiene como referencia metodológica y está preparado para trabajar con archivos Excel equivalentes ubicados en la carpeta `Datos/`.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
#cargamos el excel 
df= pd.read_excel("../Datos/maiz_cerrodelacruz.xlsx")

In [ ]:
df["Año"]=df["Año"].astype(int)

In [ ]:
#Creo índice de si usa o no tecnologia 
tec_col=["GPS", "RTK", "ISOBUS", "Siembra_variable", "Abono_variable", "Corte_tramos"]
df["Tec_indice"] = df[tec_col].sum(axis=1)


In [ ]:
Tec_indice = df.groupby("Año")["Tec_indice"].sum()
print(Tec_indice)

In [ ]:
# 4) Tabla comparativa 2021 vs 2020
base = df[df["Año"] == 2020].iloc[0]
new  = df[df["Año"] == 2021].iloc[0]

In [ ]:
metricas = [
    "Rend_t_ha", "Humedad", "Productividad_ha_h",
    "Consumo_L_h", "Consumo_L_ha",
    "Fitosanitarios_ha", "Semillas_ha", "Abono_ha"
]

In [ ]:
filas = []
for m in metricas:
    v0 = float(base[m]) #valor del 2020
    v1 = float(new[m])  #valor del 2021
    delta = v1 - v0  #cuanto cambia la metrica de un año a otro
    pct = (delta / v0) * 100 if v0 != 0 else np.nan
    filas.append([m, v0, v1, delta, pct])


comp = pd.DataFrame(filas, columns=["Metrica", "2020", "2021", "Delta", "Delta_%"])
comp_round=comp.copy()
comp_round["Delta_%"] = comp_round["Delta_%"].round(3)
print(comp_round)

In [ ]:
# 5) Normalizar a totales de parcela 
S = float(base["Superficie"])  # misma en ambos años
tot = pd.DataFrame({
    "Año": df["Año"],
    "Superficie_ha": df["Superficie"],
    "Produccion_total_t": df["Rend_t_ha"] * df["Superficie"],
    "Gasoil_total_L": df["Consumo_L_ha"] * df["Superficie"]* df["Pasadas_ha"],
    "Fitos_total_L": df["Fitosanitarios_ha"] * df["Superficie"],
})
print(tot)

In [ ]:
# 6) Gráfico 2020 vs 2021 (valores)
plot_cols = ["Rend_t_ha", "Productividad_ha_h", "Consumo_L_ha", "Fitosanitarios_ha"]
df_plot = df.set_index("Año")[plot_cols]
ax = df_plot.plot(kind="bar")
ax.set_title("Comparativa Maíz - Cerro Cruz Cati (2020 vs 2021)")
ax.set_ylabel("Valor")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("Comparativa_Cerro_de_la_Cruz.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# --- Gráfico 1: Rendimiento + Productividad (magnitudes parecidas)
cols1 = ["Rend_t_ha", "Productividad_ha_h"]
ax = df.set_index("Año")[cols1].plot(kind="bar")
ax.set_title("Rendimiento y productividad (2020 vs 2021)")
ax.set_ylabel("Valor")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# --- Gráfico 2: Consumos en L/ha (mismo tipo de unidad)
cols2 = ["Consumo_L_ha", "Fitosanitarios_ha"]
ax = df.set_index("Año")[cols2].plot(kind="bar")
ax.set_title("Consumo de insumos (2020 vs 2021)")
ax.set_ylabel("L/ha")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
#Gráfico de ahorro 

base = df[df["Año"]==2020].iloc[0]
new  = df[df["Año"]==2021].iloc[0]

ahorros = pd.Series({
    "Semillas (k/ha)": (base["Semillas_ha"] - new["Semillas_ha"]) /1000,
    "Abono (kg/ha)": (base["Abono_ha"] - new["Abono_ha"]),
})

ax = ahorros.plot(kind="bar")
ax.set_title("Ahorro de insumos por hectárea (2021 vs 2020)")
ax.set_ylabel("Ahorro (unidades/ha)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
# 7) Gráfico de mejoras (%): para consumos y fitos el "mejor" es bajar

# Calcular mejoras: positivo = mejor
metrics = ["Rend_t_ha","Humedad","Productividad_ha_h","Consumo_L_h","Consumo_L_ha","Fitosanitarios_ha","Semillas_ha","Abono_ha"]

improvement = {}
for m in metrics:
    v0 = float(base[m]); v1 = float(new[m])
    if v0 == 0:
        continue
    # Para estas, bajar es mejor
    if m in ["Consumo_L_h","Consumo_L_ha","Fitosanitarios_ha","Semillas_ha","Abono_ha","Humedad"]:
        improvement[m] = 100 * (v0 - v1) / v0
    else:
        improvement[m] = 100 * (v1 - v0) / v0

imp = pd.Series(improvement).round(3).sort_values(ascending=False)

ax = imp.plot(kind="bar")
ax.set_title("Mejoras 2021 vs 2020 (%)  [positivo = mejor]")
ax.set_ylabel("%")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()


plt.savefig("Mejoras_CerroCruz.png", dpi=300, bbox_inches="tight")


plt.show()


In [ ]:
#Dashboard de métricas

base = df[df["Año"]==2020].iloc[0]
new  = df[df["Año"]==2021].iloc[0]

info = [
    ("Rendimiento", "Rend_t_ha", "t/ha", "mas_mejor"),
    ("Productividad", "Productividad_ha_h", "ha/h", "mas_mejor"),
    ("Consumo gasoil", "Consumo_L_ha", "L/ha", "menos_mejor"),
    ("Consumo gasoil", "Consumo_L_h", "L/h", "menos_mejor"),
    ("Fitosanitarios", "Fitosanitarios_ha", "L/ha", "menos_mejor"),
    ("Semillas", "Semillas_ha", "sem/ha", "menos_mejor"),
    ("Abono", "Abono_ha", "kg/ha", "menos_mejor"),
    ("Humedad cosecha", "Humedad", "%", "menos_mejor"),
]

rows = []
for nombre, col, unidad, sentido in info:
    v0, v1 = float(base[col]), float(new[col])
    delta = v1 - v0
    pct = 100*delta/v0 if v0 else np.nan
    mejora = (100*(v0 - v1)/v0) if (sentido=="menos_mejor") else (100*(v1 - v0)/v0)
    rows.append([nombre, unidad, v0, v1, delta, pct, mejora])

res = pd.DataFrame(rows, columns=["Metrica","Unidad","2020","2021","Delta","Delta_%","Mejora_%"])
res = res.round(3)
res


In [ ]:
#Totales de campaña y ahorro acumulado
S = float(base["Superficie"])  # ha
tot = pd.DataFrame({
    "Año": df["Año"],
    "Superficie_ha": df["Superficie"],
    "Produccion_total_t": df["Rend_t_ha"] * df["Superficie"],
    "Gasoil_total_L": df["Consumo_L_ha"] * df["Superficie"],
    "Fitos_total_L": df["Fitosanitarios_ha"] * df["Superficie"],
    "Semillas_total": df["Semillas_ha"] * df["Superficie"],
    "Abono_total_kg": df["Abono_ha"] * df["Superficie"],
    "Horas_totales_est": df["Superficie"] / df["Productividad_ha_h"],
}).round(2)

tot


In [ ]:
#Gráfico Waterfall, impacto acumulado

# Filas base
base = df[df["Año"]==2020].iloc[0]
new  = df[df["Año"]==2021].iloc[0]
S = float(base["Superficie"])  # ha

# Ahorros totales (positivos = mejor)
ahorros = pd.Series({
    "Gasoil (L)": (base["Consumo_L_ha"] - new["Consumo_L_ha"]) * S,
    "Fitosanitarios (L)": (base["Fitosanitarios_ha"] - new["Fitosanitarios_ha"]) * S,
    "Abono (kg)": (base["Abono_ha"] - new["Abono_ha"]) * S,
    "Semillas (miles)": (base["Semillas_ha"] - new["Semillas_ha"]) * S / 1000,
    "Horas trabajo (h)": (S / base["Productividad_ha_h"]) - (S / new["Productividad_ha_h"]),
}).round(2)

# Preparar datos para waterfall
labels = ahorros.index.tolist()
values = ahorros.values.tolist()

cumulative = [0]
for v in values:
    cumulative.append(cumulative[-1] + v)

# Plot
fig, ax = plt.subplots(figsize=(8,5))

for i in range(len(values)):
    ax.bar(labels[i], values[i], bottom=cumulative[i])

ax.set_title("Impacto acumulado de la adopción tecnológica (2021 vs 2020)")
ax.set_ylabel("Ahorro acumulado (unidades)")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
#Waterfall en % relativo


base = df[df["Año"]==2020].iloc[0]
new  = df[df["Año"]==2021].iloc[0]

imp = pd.Series({
    "Gasoil (L/ha)": 100*(base["Consumo_L_ha"] - new["Consumo_L_ha"]) / base["Consumo_L_ha"],
    "Fitos (L/ha)": 100*(base["Fitosanitarios_ha"] - new["Fitosanitarios_ha"]) / base["Fitosanitarios_ha"],
    "Abono (kg/ha)": 100*(base["Abono_ha"] - new["Abono_ha"]) / base["Abono_ha"],
    "Semillas (sem/ha)": 100*(base["Semillas_ha"] - new["Semillas_ha"]) / base["Semillas_ha"],
    "Horas (h)": 100*((base["Superficie"]/base["Productividad_ha_h"]) - (new["Superficie"]/new["Productividad_ha_h"])) / (base["Superficie"]/base["Productividad_ha_h"]),
}).round(2)

labels = imp.index.tolist()
values = imp.values.tolist()

cumulative = [0]
for v in values:
    cumulative.append(cumulative[-1] + v)

fig, ax = plt.subplots(figsize=(9,5))
for i in range(len(values)):
    ax.bar(labels[i], values[i], bottom=cumulative[i])

ax.set_title("Impacto acumulado relativo (2021 vs 2020)")
ax.set_ylabel("Mejora acumulada (%)")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()


plt.savefig("Impacto_Acumulado_CerroCruz.png", dpi=300, bbox_inches="tight")

plt.show()

print(imp)
